In [ ]:
#importing necessary packages

import pandas as pd
import numpy as np
import matplotlib as ml
import csv

In [ ]:
#reading in the data for mon, tues, and wed

tuesday_df = pd.read_csv("data/TrafficLabelling/Tuesday-WorkingHours.pcap_ISCX.csv")
monday_df = pd.read_csv("data/TrafficLabelling/Monday-WorkingHours.pcap_ISCX.csv")
wednesday_df = pd.read_csv("data/TrafficLabelling/Wednesday-workingHours.pcap_ISCX.csv")

In [ ]:
#cleaning up the column names by stripping the surrounding whitespace

tuesday_df.columns = tuesday_df.columns.str.strip()
monday_df.columns = monday_df.columns.str.strip()
wednesday_df.columns = wednesday_df.columns.str.strip()

In [ ]:
#replace all inf w/ nans
tuesday_df = tuesday_df.replace([np.inf, -np.inf], np.nan)
monday_df = monday_df.replace([np.inf, -np.inf], np.nan)
wednesday_df = wednesday_df.replace([np.inf, -np.inf], np.nan)

In [ ]:
#dropped the duplicate col 'Fwd Header Length.1'
tuesday_df.drop('Fwd Header Length.1', axis=1, inplace=True)
monday_df.drop('Fwd Header Length.1', axis=1, inplace=True)
wednesday_df.drop('Fwd Header Length.1', axis=1, inplace=True)

In [ ]:
#monday ~ benign
mon_begnign = monday_df[monday_df['Label'] == 'BENIGN']

#get baseline port 80
mon_port_80 = mon_begnign[mon_begnign['Destination Port'] == 80]

#store the features to be used as baseline DoS port 80
features = [('Flow Packets/s', 0.995, "high", None), 
            ('Flow Duration', 0.99, "high", None), 
            ('Flow Bytes/s', 0.05, "low", ("Flow Duration", 0.99)),
            ('Flow IAT Max', 0.95, "high", None)]

#loop over the features and calculate the percentile 
port_80_results_dict = {}

for feature, level, direction, condition in features:
    baseline = "global"
    base = mon_port_80
    
    if condition is not None:
        cond_feat, cond_level = condition
        cutoff = mon_port_80[cond_feat].quantile(cond_level)
        base = mon_port_80[mon_port_80[cond_feat] > cutoff]
        baseline = f"{cond_feat} > p{int(cond_level*100)}"
    
    col = base[feature].dropna()
    port_80_results_dict[(feature, 80)] = {"value": col.quantile(level),
                                           "direction": direction,
                                           "level": level,
                                           "n": int(len(col)),
                                           "baseline": baseline}

print(port_80_results_dict)

In [ ]:
rows = []
for (metric, port), metrics_dict in port_80_results_dict.items():
    row = {
        'Metric': metric,
        'Port': port,
        **metrics_dict
    }
    rows.append(row)
df = pd.DataFrame(rows)
df.to_csv("network_metrics_monday.csv", index=False)